# 02 · Análisis exploratorio de datos (EDA)

**Proyecto semestral IMT2200, Grupo 6:** *¿Mide el NEM lo mismo para todos?*

**Entrada:** `data/processed/panel_completo.parquet` (generado por
`01_carga_y_limpieza.ipynb`). **Salidas:** `data/processed/muestra.parquet` (para el
modelamiento) y las figuras en `imgs/`.

El EDA está ordenado según las tres preguntas guía de la propuesta, precedidas por
secciones de preparación que responden a la retroalimentación de la Entrega 1:

| Sección | Contenido |
|---|---|
| 0 | Configuración y carga del panel |
| 1 | Calidad de datos y comparabilidad entre años |
| 2 | Muestra analítica y definición de la brecha |
| 3 | Sensibilidad a la prueba PAES elegida |
| 4 | Pregunta 1: ¿cómo se relacionan NEM y PAES? |
| 5 | Pregunta 2: brecha según dependencia y nivel socioeconómico |
| 6 | Nivel establecimiento (insumo para el modelo jerárquico) |
| 7 | Pregunta 3: brecha y etapas del acceso |
| 8 | Evolución 2023-2026 |
| 9 | Síntesis de hallazgos |

Las celdas con **VERIFICAR** en los comentarios dependen de un supuesto que hay que
confirmar mirando su salida.

## 0. Configuración

In [ ]:
# 0. Rutas, librerías, módulos del proyecto (src/) y carga del panel.
import sys
from pathlib import Path

# Raíz del proyecto (funciona si el notebook se abre desde notebooks/ o
# desde la raíz) para poder importar los módulos de src/.
RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import (ARCHIVO_MUESTRA, ARCHIVO_PANEL, CRITERIO_RENDICION,
                        DEPENDENCIA_ETIQUETAS, ELECTIVAS, ESTADO_SELECCIONADO,
                        ESTADOS_VALIDOS, INCLUIR_AMBAS,
                        MIN_ESTUDIANTES_COLEGIO, MIN_N_CELDA,
                        ORDEN_DEPENDENCIA, PRUEBAS_PAES,
                        RBD_SIN_ESTABLECIMIENTO)
from src.estadistica import (correlaciones_anio, diagnostico_columnas,
                             icc_anova, resumen, tasas_acceso, v_de_cramer)
from src.graficos import aplicar_estilo, graficar_medias, mostrar
from src.muestra import (agregar_variables, calcular_brechas,
                         construir_muestra, puntaje_valido)

aplicar_estilo()
panel_completo = pd.read_parquet(ARCHIVO_PANEL)
print(f'Panel cargado: {len(panel_completo)} filas, '
      f'{panel_completo.shape[1]} columnas, '
      f'años {sorted(panel_completo["ANIO"].unique())}')

## 1 Calidad de datos y comparabilidad entre años

Antes de mirar la brecha se revisa que las variables signifiquen lo mismo en los
cuatro procesos (punto pedido explícitamente en la retroalimentación): tipos de
datos, faltantes, ceros usados como "no rindió", rangos de puntaje y códigos de
las variables categóricas.

In [ ]:
# 1.a Tipos de datos de las columnas clave (deberían ser todas numéricas
# después de la Decisión 4, salvo RAMA_EDUCACIONAL).
COLUMNAS_DIAGNOSTICO: list[str] = [
    'ANYO_DE_EGRESO', 'PROMEDIO_NOTAS', 'PTJE_NEM', 'CLEC_REG_ACTUAL',
    'MATE1_REG_ACTUAL', 'HCSOC_REG_ACTUAL', 'CIEN_REG_ACTUAL', 'RBD',
    'DEPENDENCIA', 'RAMA_EDUCACIONAL', 'INGRESO_PERCAPITA_GRUPO_FA',
]
print('Tipos de datos en panel_completo:')
print(panel_completo[COLUMNAS_DIAGNOSTICO].dtypes.to_string())

In [ ]:
# 1.b Porcentaje de nulos, de ceros y rango (mín-máx) por año.
# Un % de ceros alto en un puntaje indica que 0 codifica "no rindió".
diag = diagnostico_columnas(panel_completo, COLUMNAS_DIAGNOSTICO)
for medida in ['pct_nulo', 'pct_cero', 'min', 'max']:
    print(f'\n--- {medida} (filas: columna, columnas: año) ---')
    display(diag.pivot(index='columna', columns='ANIO', values=medida)
            .round(1))

In [ ]:
# 1.c Códigos presentes en las variables categóricas, por año.
# VERIFICAR: que los códigos sean los mismos en los 4 procesos y que todos
# los de DEPENDENCIA estén en DEPENDENCIA_ETIQUETAS.
for col in ['DEPENDENCIA', 'RAMA_EDUCACIONAL', 'INGRESO_PERCAPITA_GRUPO_FA']:
    print(f'\n--- {col} (conteo por año; <NA> = faltante) ---')
    display(pd.crosstab(panel_completo[col].astype('string').fillna('<NA>'),
                        panel_completo['ANIO']))

codigos = set(panel_completo['DEPENDENCIA'].dropna().astype(int).unique())
sin_mapa = codigos - set(DEPENDENCIA_ETIQUETAS)
print(f'Códigos de DEPENDENCIA sin etiqueta: {sorted(sin_mapa) or "ninguno"}')

In [ ]:
# 1.d ¿Cuál es la "primera rendición"? Entre egresados recientes, cuántos
# rindieron cada prueba en invierno (INV, junio) y/o regular (REG, diciembre)
# del proceso actual, y cuántos tienen rendiciones del proceso anterior.
# VERIFICAR: si muchos rindieron INV y REG, la decisión de usar REG como
# "primera" debe justificarse (ver CRITERIO_RENDICION en 2).
recientes = panel_completo[panel_completo['egresado_reciente']]
filas = []
for prueba in ['CLEC', 'MATE1', 'HCSOC', 'CIEN']:
    reg = puntaje_valido(recientes[f'{prueba}_REG_ACTUAL'])
    inv = puntaje_valido(recientes[f'{prueba}_INV_ACTUAL'])
    ant = (puntaje_valido(recientes[f'{prueba}_REG_ANTERIOR'])
           | puntaje_valido(recientes[f'{prueba}_INV_ANTERIOR']))
    conteo = pd.DataFrame({'solo_REG': reg & ~inv, 'solo_INV': inv & ~reg,
                           'REG_e_INV': reg & inv, 'proceso_anterior': ant})
    tabla = conteo.groupby(recientes['ANIO']).sum()
    tabla.insert(0, 'prueba', PRUEBAS_PAES[prueba])
    filas.append(tabla)
print(f'Egresados recientes: {len(recientes)} (4 años)')
display(pd.concat(filas).reset_index().set_index(['prueba', 'ANIO']))

## 2. Muestra analítica y definición de la brecha

La muestra son los egresados del año inmediatamente anterior al proceso, con NEM
válido y que rindieron **al menos una** electiva. Se forman tres grupos: solo
Ciencias, solo Historia y **Ambas** (PAES electiva = promedio de los dos puntajes).
Cada estudiante se compara solo con su grupo, porque Ciencias e Historia son pruebas
distintas rendidas por poblaciones distintas.

Las brechas se definen una sola vez, en `src/muestra.py`:

- **Brecha simple** `NEM_z - PAES_z`, con z dentro de cada año y grupo de electiva.
  `> 0` significa una posición relativa más alta en NEM que en la PAES.
- **Brecha condicional** `NEM_z - r·PAES_z`: NEM observado menos el NEM esperado dada
  la PAES. Responde *"a igual PAES, ¿distinto NEM?"* y corrige la regresión a la
  media de la brecha simple (ver 5.f).

Ambas tienen media 0 en cada año y grupo, así que lo comparable son las
**diferencias entre grupos** (dependencia, decil, colegio), no su nivel entre años.

In [ ]:
# 2.a Muestra analítica (filtros definidos en src/muestra.py).
# Parámetros en src/config.py: CRITERIO_RENDICION e INCLUIR_AMBAS.
muestra, embudo = construir_muestra(panel_completo, CRITERIO_RENDICION,
                                    INCLUIR_AMBAS)
print(f'Criterio de rendición: {CRITERIO_RENDICION} | '
      f'incluir ambas electivas: {INCLUIR_AMBAS}')
print('Embudo de filtros (n por año):')
display(embudo)
print(f'Muestra analítica: {len(muestra)} estudiantes en 4 años.')
print('Estudiantes por grupo de electiva:')
print(muestra['electiva'].value_counts().to_string())

In [ ]:
# 2.b Brecha simple, brecha condicional y brechas alternativas (Lectora y
# M1). La definición está documentada en src/muestra.py:calcular_brechas.
muestra = calcular_brechas(muestra)
print('Brecha simple por año y electiva (media 0 por diseño):')
display(muestra.groupby(['ANIO', 'electiva'])['brecha']
        .agg(['count', 'mean', 'std']).round(3))

In [ ]:
# 2.c Variables explicativas con etiquetas legibles (src/muestra.py).
n_sin_rbd = (muestra['RBD'] == RBD_SIN_ESTABLECIMIENTO).sum()
muestra = agregar_variables(muestra)
print(f'RBD {RBD_SIN_ESTABLECIMIENTO}: {n_sin_rbd} estudiantes '
      f'(se excluyen del análisis por colegio)')

print('Distribución de la muestra por dependencia (%):')
print((100 * muestra['dependencia'].value_counts(normalize=True, sort=False))
      .round(1).to_string())
no_responde = (muestra['decil'] == 'No responde').mean()
print(f'\nNo responde ingreso: {no_responde:.1%}')

## 3 Sensibilidad a la prueba PAES elegida

La retroalimentación pregunta si la electiva es la mejor referencia frente a
Competencia Lectora o Matemática 1, que tienen mayor cobertura. Esta sección compara
las tres versiones de la brecha: si ordenan a los grupos de la misma forma, la
elección no cambia las conclusiones y se puede justificar por cobertura o
interpretación; si difieren, hay que elegir explícitamente qué se quiere medir.

In [ ]:
# 3.a Cobertura y correlación de NEM con cada prueba, por año.
cobertura = muestra[['CLEC_1ra', 'MATE1_1ra', 'PAES_electiva']].notna().mean()
print('Cobertura dentro de la muestra (%):')
print((100 * cobertura).round(1).to_string())

correlaciones = muestra.groupby('ANIO').apply(correlaciones_anio,
                                              include_groups=False)
print('\nCorrelación de Pearson entre NEM y cada prueba:')
display(correlaciones.round(3))

In [ ]:
# 3.b ¿Las tres brechas ordenan igual a los grupos de dependencia?
versiones = {'brecha': 'Electiva', 'brecha_CLEC': 'Lectora',
             'brecha_MATE1': 'Matemática 1',
             'brecha_cond': 'Electiva condicional'}
print('Correlación entre versiones de la brecha (nivel estudiante):')
display(muestra[list(versiones)].corr().round(3))

medias = (muestra.groupby('dependencia', observed=True)[list(versiones)]
          .mean().rename(columns=versiones))
display(medias.round(3))

ax = medias.plot(kind='bar', rot=0)
ax.axhline(0, color='black', linewidth=0.8)
ax.set(title='Brecha media por dependencia según la prueba de referencia',
       xlabel='', ylabel='Brecha media (DE)')
plt.tight_layout()
mostrar('eda_3b')

## 4 Pregunta 1: ¿cómo se relacionan el NEM y la PAES?

In [ ]:
# 4.a Descriptivos de notas, NEM y PAES electiva por año y electiva.
descriptivos = muestra.groupby(['ANIO', 'electiva'])[
    ['PROMEDIO_NOTAS', 'PTJE_NEM', 'PAES_electiva']
].describe(percentiles=[0.25, 0.5, 0.75])
# Se muestran solo media, DE y cuartiles para no saturar la salida.
cols = [(v, e) for v in ['PROMEDIO_NOTAS', 'PTJE_NEM', 'PAES_electiva']
        for e in ['mean', 'std', '25%', '50%', '75%']]
display(descriptivos[cols].round(2))

In [ ]:
# 4.b Distribución de NEM y PAES electiva, una curva por año.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for anio, grupo in muestra.groupby('ANIO'):
    axes[0].hist(grupo['PTJE_NEM'], bins=60, histtype='step',
                 density=True, label=str(anio))
    axes[1].hist(grupo['PAES_electiva'], bins=60, histtype='step',
                 density=True, label=str(anio))
axes[0].set(title='Puntaje NEM', xlabel='Puntaje', ylabel='Densidad')
axes[1].set(title='PAES electiva (Ciencias o Historia)', xlabel='Puntaje')
axes[1].legend(title='Proceso')
plt.tight_layout()
mostrar('eda_4b')

In [ ]:
# 4.c NEM_z vs PAES_z. Sobre la diagonal la brecha es 0; bajo ella,
# brecha > 0 (NEM relativamente más alto que la PAES).
presentes = [e for e in ELECTIVAS if (muestra['electiva'] == e).any()]
fig, axes = plt.subplots(1, len(presentes), figsize=(6 * len(presentes), 5),
                         sharex=True, sharey=True)
for ax, electiva in zip(np.atleast_1d(axes), presentes):
    datos = muestra[muestra['electiva'] == electiva]
    hb = ax.hexbin(datos['PAES_z'], datos['NEM_z'], gridsize=60,
                   bins='log', cmap='viridis', mincnt=1)
    ax.plot([-4, 4], [-4, 4], color='red', linewidth=1, label='brecha = 0')
    r = datos['NEM_z'].corr(datos['PAES_z'])
    ax.set(title=f'{electiva} (r = {r:.2f}, n = {len(datos)})',
           xlabel='PAES electiva (z)', ylabel='NEM (z)',
           xlim=(-4, 4), ylim=(-4, 4))
    ax.legend(loc='upper left')
fig.colorbar(hb, ax=axes, label='log10(n)')
mostrar('eda_4c')

In [ ]:
# 4.d Distribución de la brecha y proporción de casos con brechas grandes.
fig, ax = plt.subplots()
for electiva, grupo in muestra.groupby('electiva'):
    ax.hist(grupo['brecha'], bins=80, histtype='step', density=True,
            label=electiva)
ax.axvline(0, color='black', linewidth=0.8)
ax.set(title='Distribución de la brecha NEM - PAES',
       xlabel='Brecha (DE)', ylabel='Densidad', xlim=(-5, 5))
ax.legend()
mostrar('eda_4d')

print(f"|brecha| > 1 DE: {(muestra['brecha'].abs() > 1).mean():.1%}")
print(f"brecha > 1 DE:  {(muestra['brecha'] > 1).mean():.1%}")
print(f"brecha < -1 DE: {(muestra['brecha'] < -1).mean():.1%}")

In [ ]:
# 4.e Conversión de promedio de notas a puntaje NEM según modalidad.
# La tabla de conversión DEMRE depende de la modalidad (HC / TP), así que
# un mismo promedio puede dar distinto NEM. Si las nubes no coinciden,
# parte de la brecha por modalidad es un efecto de escala, no de notas.
rng = np.random.default_rng(2026)
sub = muestra.iloc[rng.choice(len(muestra), size=min(30_000, len(muestra)),
                              replace=False)]
fig, ax = plt.subplots()
for modalidad, grupo in sub.groupby('modalidad'):
    ax.scatter(grupo['PROMEDIO_NOTAS'], grupo['PTJE_NEM'], s=3, alpha=0.3,
               label=modalidad)
ax.set(title='Promedio de notas vs puntaje NEM (submuestra)',
       xlabel='Promedio de notas', ylabel='Puntaje NEM')
ax.legend(markerscale=4)
mostrar('eda_4e')

## 5 Pregunta 2: brecha según dependencia y nivel socioeconómico

Además de la brecha, se muestran sus dos componentes (NEM_z y PAES_z): una brecha
positiva puede deberse a un NEM alto, a una PAES baja o a ambos, y la
interpretación cambia según el caso.

In [ ]:
# 5.a Brecha por dependencia: tabla, boxplot y descomposición.
print('Brecha por dependencia:')
display(resumen(muestra, 'dependencia', 'brecha'))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
por_dependencia = list(muestra.groupby('dependencia', observed=True))
grupos = [g['brecha'].dropna() for _, g in por_dependencia]
etiquetas = [str(d) for d, _ in por_dependencia]
axes[0].boxplot(grupos, showfliers=False)
axes[0].set_xticks(range(1, len(etiquetas) + 1), etiquetas, rotation=20)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set(title='Brecha por dependencia (sin atípicos)',
            ylabel='Brecha (DE)')

componentes = muestra.groupby('dependencia', observed=True)[
    ['NEM_z', 'PAES_z', 'brecha']].mean()
componentes.plot(kind='bar', ax=axes[1], rot=20)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set(title='Componentes de la brecha por dependencia',
            xlabel='', ylabel='Media (DE)')
plt.tight_layout()
mostrar('eda_5a')

In [ ]:
# 5.b Brecha media por decil de ingreso, una línea por dependencia.
# Permite ver si el gradiente socioeconómico existe dentro de cada tipo de
# establecimiento o si solo refleja la composición de cada dependencia.
fig, ax = plt.subplots(figsize=(10, 5))
for dependencia in ['Municipal', 'SLEP', 'Part. subvencionado',
                    'Part. pagado']:
    datos = muestra[muestra['dependencia'] == dependencia]
    if len(datos) == 0:
        continue
    graficar_medias(resumen(datos, 'decil', 'brecha'), ax, dependencia)
ax.axhline(0, color='black', linewidth=0.8)
ax.set(title='Brecha media por decil de ingreso y dependencia (IC 95%)',
       xlabel='Decil de ingreso per cápita', ylabel='Brecha media (DE)')
ax.legend()
mostrar('eda_5b')

In [ ]:
# 5.c Mapa de calor dependencia x decil (se ocultan celdas con n bajo).
tabla_media = muestra.pivot_table(index='dependencia', columns='decil',
                                  values='brecha', aggfunc='mean',
                                  observed=True)
tabla_n = muestra.pivot_table(index='dependencia', columns='decil',
                              values='brecha', aggfunc='count',
                              observed=True)
tabla_media = tabla_media.where(tabla_n >= MIN_N_CELDA)

fig, ax = plt.subplots(figsize=(11, 4))
limite = np.nanmax(np.abs(tabla_media.to_numpy()))
im = ax.imshow(tabla_media, cmap='RdBu_r', vmin=-limite, vmax=limite,
               aspect='auto')
ax.set_xticks(range(tabla_media.shape[1]), tabla_media.columns)
ax.set_yticks(range(tabla_media.shape[0]), tabla_media.index)
for i in range(tabla_media.shape[0]):
    for j in range(tabla_media.shape[1]):
        valor = tabla_media.iat[i, j]
        if pd.notna(valor):
            ax.text(j, i, f'{valor:.2f}', ha='center', va='center',
                    fontsize=8)
fig.colorbar(im, ax=ax, label='Brecha media (DE)')
ax.set(title='Brecha media por dependencia y decil de ingreso',
       xlabel='Decil')
plt.tight_layout()
mostrar('eda_5c')

In [ ]:
# 5.d Composición socioeconómica de cada dependencia: anticipa la
# colinealidad que luego medirá el VIF.
composicion = pd.crosstab(muestra['dependencia'], muestra['decil'],
                          normalize='index')
ax = (100 * composicion).plot(kind='barh', stacked=True, figsize=(11, 4),
                              colormap='viridis', width=0.8)
ax.set(title='Distribución de deciles dentro de cada dependencia',
       xlabel='% de estudiantes', ylabel='')
ax.legend(title='Decil', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
mostrar('eda_5d')

declaran = muestra[muestra['decil'] != 'No responde']
v = v_de_cramer(pd.crosstab(declaran['dependencia'],
                            declaran['decil'].astype(str)))
print(f'V de Cramér dependencia x decil (sin no respuesta): {v:.3f}')

In [ ]:
# 5.e Brecha según modalidad y sexo (posibles variables de control).
for variable in ['modalidad', 'sexo']:
    print(f'\nBrecha por {variable}:')
    display(resumen(muestra, variable, 'brecha'))

### 5.f Brecha simple vs brecha condicional (regresión a la media)

La brecha simple `NEM_z - PAES_z` tiene un problema conocido de las diferencias de
puntajes: como la correlación NEM-PAES es menor que 1 (r entre 0,5 y 0,57), un grupo con
PAES alta tendrá en promedio una brecha negativa **aunque el NEM mida exactamente lo
mismo para todos**, solo por regresión a la media. Por ejemplo, en 5.a Part. pagado
tiene PAES_z ≈ +1,1 y NEM_z ≈ +0,6, que es aproximadamente lo que se esperaría con
r ≈ 0,55.

La **brecha condicional** responde más directamente la pregunta del título: *a igual
puntaje PAES, ¿los estudiantes de distintos grupos tienen distinto NEM?* Es
equivalente a modelar `NEM_z ~ PAES_z + dependencia + ...`, que es la forma natural
del modelo jerárquico. Además, el NEM tiene un techo (promedio 7,0 = 1000 puntos)
visible en el gráfico de 4.c, que limita la brecha de quienes tienen PAES alta.

In [ ]:
# 5.f.1 Brecha simple vs condicional por dependencia.
comparacion = muestra.groupby('dependencia', observed=True).agg(
    n=('brecha', 'size'),
    NEM_z=('NEM_z', 'mean'),
    PAES_z=('PAES_z', 'mean'),
    brecha_simple=('brecha', 'mean'),
    brecha_condicional=('brecha_cond', 'mean'),
)
print('Medias por dependencia (DE):')
display(comparacion.round(3))

ax = comparacion[['brecha_simple', 'brecha_condicional']].plot(
    kind='bar', rot=20, figsize=(10, 4.5))
ax.axhline(0, color='black', linewidth=0.8)
ax.set(title='Brecha simple vs condicional por dependencia', xlabel='',
       ylabel='Media (DE)')
plt.tight_layout()
mostrar('eda_5f1')

In [ ]:
# 5.f.2 Brecha condicional por decil, una línea por dependencia.
fig, ax = plt.subplots(figsize=(10, 5))
for dependencia in ['Municipal', 'SLEP', 'Part. subvencionado',
                    'Part. pagado']:
    datos = muestra[muestra['dependencia'] == dependencia]
    if len(datos) == 0:
        continue
    graficar_medias(resumen(datos, 'decil', 'brecha_cond'), ax, dependencia)
ax.axhline(0, color='black', linewidth=0.8)
ax.set(title='Brecha condicional por decil y dependencia (IC 95%)',
       xlabel='Decil de ingreso per cápita',
       ylabel='NEM_z observado - esperado (DE)')
ax.legend()
mostrar('eda_5f2')

In [ ]:
# 5.f.3 Techo del NEM: % de estudiantes con NEM máximo (promedio 7,0).
# Si un grupo concentra muchos casos en el techo, su brecha está acotada
# hacia arriba y conviene considerarlo al interpretar (o en el modelo).
NEM_MAXIMO: float = muestra['PTJE_NEM'].max()
muestra['nem_en_techo'] = muestra['PTJE_NEM'] >= NEM_MAXIMO
terciles = pd.qcut(muestra['PAES_z'], 3,
                   labels=['PAES baja', 'PAES media', 'PAES alta'])
techo = 100 * (muestra.groupby(['dependencia', terciles],
                               observed=True)['nem_en_techo'].mean()
               .unstack())
print(f'NEM máximo observado: {NEM_MAXIMO:.0f}')
print(f"% en el techo (total): {100 * muestra['nem_en_techo'].mean():.1f}%")
print('% en el techo por dependencia y tercil de PAES:')
display(techo.round(1))

## 6 Nivel establecimiento (insumo para el modelo jerárquico)

El modelo jerárquico tendrá estudiantes (nivel 1) dentro de establecimientos (nivel 2,
identificados por `RBD`). Esta sección mide cuánta de la variación de la brecha ocurre
*entre* colegios (coeficiente de correlación intraclase, ICC) y describe las variables
candidatas a nivel colegio: dependencia y composición socioeconómica.

In [ ]:
# 6.a Agregación por establecimiento (4 años juntos).
colegios = muestra.groupby('RBD').agg(
    n=('brecha', 'size'),
    brecha_media=('brecha', 'mean'),
    nem_z_media=('NEM_z', 'mean'),
    paes_z_media=('PAES_z', 'mean'),
    decil_medio=('decil_num', 'mean'),
    pct_decil_1a3=('decil_num', lambda s: (s <= 3).sum() / s.notna().sum()
                   if s.notna().any() else np.nan),
    dependencia=('dependencia',
                 lambda s: s.mode().iat[0] if s.notna().any() else np.nan),
    n_dependencias=('DEPENDENCIA', 'nunique'),
)
print(f'Establecimientos en la muestra: {len(colegios)}')
print('Estudiantes por establecimiento:')
print(colegios['n'].describe().round(1).to_string())
print(f"Con al menos {MIN_ESTUDIANTES_COLEGIO} estudiantes: "
      f"{(colegios['n'] >= MIN_ESTUDIANTES_COLEGIO).sum()}")
# VERIFICAR: un RBD con más de una dependencia indica cambio de
# administración entre años (p. ej., traspaso a SLEP).
print(f"RBD con más de una dependencia: "
      f"{(colegios['n_dependencias'] > 1).sum()}")

In [ ]:
# 6.b ICC (estimador ANOVA, src/estadistica.py): proporción de la varianza
# que ocurre entre colegios.
VARIABLES_ICC: list[str] = ['brecha', 'brecha_cond', 'NEM_z', 'PAES_z']
iccs = pd.DataFrame({
    anio: {v: icc_anova(g, v, 'RBD') for v in VARIABLES_ICC}
    for anio, g in muestra.groupby('ANIO')
})
iccs['4 años'] = [icc_anova(muestra, v, 'RBD') for v in VARIABLES_ICC]
print('ICC por variable (fila) y año (columna):')
display(iccs.round(3))

In [ ]:
# 6.c Brecha media por colegio vs nivel socioeconómico medio del colegio.
grandes = colegios[colegios['n'] >= MIN_ESTUDIANTES_COLEGIO]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for dependencia, grupo in grandes.groupby('dependencia', observed=True):
    axes[0].scatter(grupo['decil_medio'], grupo['brecha_media'],
                    s=np.sqrt(grupo['n']) * 2, alpha=0.5, label=dependencia)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set(title=f'Colegios con n >= {MIN_ESTUDIANTES_COLEGIO}',
            xlabel='Decil de ingreso medio del colegio',
            ylabel='Brecha media del colegio (DE)')
axes[0].legend(markerscale=0.7, fontsize=8)

etiquetas = [d for d in ORDEN_DEPENDENCIA
             if d in set(grandes['dependencia'].dropna())]
axes[1].boxplot([grandes.loc[grandes['dependencia'] == d, 'brecha_media']
                 for d in etiquetas], showfliers=False)
axes[1].set_xticks(range(1, len(etiquetas) + 1), etiquetas, rotation=20)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set(title='Brecha media del colegio por dependencia',
            ylabel='Brecha media (DE)')
plt.tight_layout()
mostrar('eda_6c')

## 7 Pregunta 3: brecha y etapas del acceso

Siguiendo la retroalimentación, el acceso se separa en etapas: **postuló** (aparece en
PS), **postulación válida** (seleccionado o lista de espera en alguna preferencia;
requiere completar `ESTADOS_VALIDOS` en la Decisión 2) y **matriculado** (aparece en
M). Las tasas se muestran sobre toda la muestra y condicionadas a la etapa anterior.

Advertencia de interpretación: NEM y PAES entran directamente al puntaje ponderado,
así que parte de cualquier asociación brecha-acceso es mecánica. La celda 7.c
compara estudiantes con PAES similar para separar parcialmente ese efecto.

In [ ]:
# 7.a Tasas por etapa según dependencia y decil (src/estadistica.py).
print(f'Códigos usados -> seleccionado: {ESTADO_SELECCIONADO}, '
      f'válida (seleccionado o espera): {sorted(ESTADOS_VALIDOS)}')
for grupo in ['dependencia', 'decil']:
    print(f'\nTasas de acceso (%) por {grupo}:')
    display(tasas_acceso(muestra, grupo))

In [ ]:
# 7.b Embudo de acceso por dependencia.
etapas = tasas_acceso(muestra, 'dependencia')[
    ['postuló', 'seleccionado', 'matriculado']]
ax = etapas.plot(kind='bar', rot=20, figsize=(10, 4.5))
ax.set(title='Etapas del acceso por dependencia', xlabel='',
       ylabel='% de la muestra')
plt.tight_layout()
mostrar('eda_7b')

In [ ]:
# 7.c Matrícula según quintil de brecha, dentro de terciles de PAES_z.
# Si la tasa varía con la brecha incluso entre estudiantes con PAES similar,
# la asociación no se explica solo por el nivel de PAES.
muestra['quintil_brecha'] = pd.qcut(muestra['brecha'], 5,
                                    labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
muestra['tercil_paes'] = pd.qcut(muestra['PAES_z'], 3,
                                 labels=['PAES baja', 'PAES media',
                                         'PAES alta'])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (titulo, datos) in zip(axes, [
        ('Toda la muestra', muestra),
        ('Solo quienes postularon', muestra[muestra['postulo']])]):
    tasas = datos.pivot_table(index='quintil_brecha', columns='tercil_paes',
                              values='matriculado', aggfunc='mean',
                              observed=True)
    (100 * tasas).plot(marker='o', ax=ax)
    ax.set(title=f'Matrícula por quintil de brecha ({titulo.lower()})',
           xlabel='Quintil de brecha (Q5 = NEM relativamente más alto)',
           ylabel='% matriculado')
plt.tight_layout()
mostrar('eda_7c')

## 8 Evolución 2023-2026

Como la brecha está estandarizada por año, su promedio general es 0 en cada proceso;
lo que sí puede cambiar en el tiempo es la **diferencia entre grupos**. Para mirar
niveles absolutos se usan las variables sin estandarizar (promedio de notas, NEM,
PAES). Con cuatro años, esto describe brechas observadas en el periodo, no una
tendencia de largo plazo de inflación de notas (pregunta 6 de la retroalimentación).

In [ ]:
# 8.a Niveles absolutos por año (variables sin estandarizar).
muestra['promedio_6_o_mas'] = muestra['PROMEDIO_NOTAS'] >= 6.0
niveles = muestra.groupby('ANIO').agg(
    promedio_notas=('PROMEDIO_NOTAS', 'mean'),
    pct_promedio_6=('promedio_6_o_mas', 'mean'),
    nem=('PTJE_NEM', 'mean'),
    lectora=('CLEC_1ra', 'mean'),
    m1=('MATE1_1ra', 'mean'),
)
niveles['pct_promedio_6'] *= 100
print('Medias anuales en la muestra (sin estandarizar):')
display(niveles.round(2))
print('Media de PAES electiva por año y electiva:')
display(muestra.pivot_table(index='ANIO', columns='electiva',
                            values='PAES_electiva', aggfunc='mean').round(1))

In [ ]:
# 8.b Diferencias entre dependencias a lo largo del tiempo.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for dependencia in ['Municipal', 'SLEP', 'Part. subvencionado',
                    'Part. pagado']:
    datos = muestra[muestra['dependencia'] == dependencia]
    if len(datos) == 0:
        continue
    graficar_medias(resumen(datos, 'ANIO', 'brecha'), axes[0], dependencia)
    pct = 100 * datos.groupby('ANIO')['promedio_6_o_mas'].mean()
    axes[1].plot(pct.index.astype(str), pct.values, marker='o',
                 label=dependencia)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set(title='Brecha media por año y dependencia (IC 95%)',
            xlabel='Proceso', ylabel='Brecha media (DE)')
axes[1].set(title='% con promedio de notas >= 6,0',
            xlabel='Proceso', ylabel='%')
axes[0].legend(fontsize=8)
plt.tight_layout()
mostrar('eda_8b')

## Guardar la muestra para el modelamiento

In [ ]:
# Guarda la muestra con brechas y variables (data/processed/ en .gitignore).
muestra.to_parquet(ARCHIVO_MUESTRA, index=False)
print(f'Guardado {ARCHIVO_MUESTRA.name}: {len(muestra)} filas, '
      f'{muestra.shape[1]} columnas')

## 9 Síntesis de hallazgos del EDA

*(Completar después de correr las celdas.)*

**Calidad y muestra.** Faltantes y códigos relevantes, tamaño final de la muestra y
pasos del embudo que más observaciones eliminan.

**Pregunta 1.** Correlación NEM-PAES por electiva; forma de la distribución de la
brecha; diferencias de escala NEM por modalidad.

**Pregunta 2.** Orden de las dependencias según brecha y cuál componente (NEM o PAES)
la explica; gradiente por decil dentro de cada dependencia; grado de asociación
entre dependencia y decil (V de Cramér).

**Nivel colegio.** ICC de la brecha y qué implica para el modelo jerárquico.

**Pregunta 3.** Tasas por etapa y relación brecha-matrícula a igual nivel de PAES.

**Decisiones que salen del EDA.** Prueba de referencia, criterio de rendición,
tratamiento de quienes rinden ambas electivas, y variables a nivel estudiante vs.
nivel establecimiento.